In [10]:
import torch
import joblib
scaler=joblib.load('scaler.pkl')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [20]:
data_to_transform=torch.load('/Users/robinhu/Desktop/torchdata/folkdataset.pt')['folk_feature']
final=torch.tensor(scaler.transform(data_to_transform.reshape(data_to_transform.shape[0],-1)).reshape(-1,80,616)).float()

In [13]:
model_path = '/Users/robinhu/Desktop/torchdata/CNN.pt'

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class OptimizedCNNLSTMClassifier(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2)), 
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2)), 
            
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2)),  
            
      
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, None))  # [256, 1, 77]
        )
        
        # LSTM part
        # [batch, 77, 256]
        self.lstm = nn.LSTM(
            input_size=256,          
            hidden_size=128,         
            num_layers=2,           
            bidirectional=True,      
            batch_first=True,        
            dropout=dropout_rate     
        )
        
        # Attention
        self.attention = nn.Sequential(
            nn.Linear(256, 128),     
            nn.Tanh(),               
            nn.Linear(128, 1)        
        )

        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),     
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)        
        )
        
        self._init_weights()
        
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        cnn_out = self.cnn(x)
        cnn_out = cnn_out.squeeze(2)
        cnn_out = cnn_out.permute(0, 2, 1)
        lstm_out, _ = self.lstm(cnn_out)
        attention_weights = F.softmax(self.attention(lstm_out), dim=1)
        weighted_output = torch.sum(attention_weights * lstm_out, dim=1)
        logits = self.classifier(weighted_output)
        
        return logits
    
    def predict_proba(self, x, threshold=0.5):
        with torch.no_grad():
            self.eval()  
            logits = self.forward(x)
            probabilities = torch.sigmoid(logits)  
            return probabilities
    
    def predict(self, x, threshold=0.5):
        probabilities = self.predict_proba(x)
        predictions = (probabilities > threshold).float()
        return predictions
    
    def get_attention_map(self, x):
        with torch.no_grad():
            self.eval()
            x = x.unsqueeze(1)
            cnn_out = self.cnn(x)
            cnn_out = cnn_out.squeeze(2).permute(0, 2, 1)
            lstm_out, _ = self.lstm(cnn_out)
            attention_weights = F.softmax(self.attention(lstm_out), dim=1)
            return attention_weights.squeeze(-1)  # [batch, 77]

In [15]:
def load_trained_model(model_path):
    model = OptimizedCNNLSTMClassifier()
    loaded_data = torch.load(model_path, map_location='cpu', weights_only=False)
    model.load_state_dict(loaded_data)
    model.eval()
    
    return model

In [16]:
model=load_trained_model(model_path)

In [17]:
def folk_to_jazz_transform(model, jazz_spec, num_iterations,
                                 content_weight, lr=0.05, 
                                 target_logit=-2.0):

    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    
    device = next(model.parameters()).device
    if len(jazz_spec.shape) == 2:
        jazz_spec = jazz_spec.unsqueeze(0)
    
    original = jazz_spec.to(device) if isinstance(jazz_spec, torch.Tensor) \
              else torch.FloatTensor(jazz_spec).to(device)
    with torch.no_grad():
        initial_logit = model(original)
        initial_prob = torch.sigmoid(initial_logit).item()
        print(f"initial probability - jazz: {1-initial_prob:.4f}, folk: {initial_prob:.4f}")
    
    generated = original.clone().detach()
    generated.requires_grad_(True)
    optimizer = torch.optim.Adam([generated], lr=lr, betas=(0.9, 0.999))
    
    history = {
        'logits': [],
        'jazz_prob': [],
        'folk_prob': [],
        'content_loss': [],
        'style_loss': [],
        'total_loss': []
    }
    
    for i in range(num_iterations):
        optimizer.zero_grad()
        logit = model(generated)
        folk_prob = torch.sigmoid(logit)
        jazz_prob = 1 - folk_prob
        style_loss = F.mse_loss(logit, torch.tensor([target_logit], device=device))
        content_loss = F.mse_loss(generated, original)
        total_loss = style_loss + content_weight * content_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_([generated], max_norm=1.0)
        optimizer.step()
        
        history['logits'].append(logit.item())
        history['jazz_prob'].append(jazz_prob.item())
        history['folk_prob'].append(folk_prob.item())
        history['content_loss'].append(content_loss.item())
        history['style_loss'].append(style_loss.item())
        history['total_loss'].append(total_loss.item())
        
        if (i + 1) % 20 == 0:
            print(f"Iter {i+1:3d}: "
                  f"Jazz={jazz_prob.item():.4f}, "
                  f"Folk={folk_prob.item():.4f}, "
                  f"StyleLoss={style_loss.item():.4f}")
    
    with torch.no_grad():
        final_logit = model(generated)
        final_prob = torch.sigmoid(final_logit).item()
    print(f"jazz: {1-final_prob:.4f}, folk: {final_prob:.4f}")
    print(f"Content Loss: {history['content_loss'][-1]:.6f}")
    return generated.detach(), history


def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [27]:
def batch_style_transfer_test(model, folk_samples, num_samples=2207, 
                             content_weight=0.2, iterations=30):
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    
    device = next(model.parameters()).device
    results = []
    transformed_total=[]
    
    for i in range(min(num_samples, len(folk_samples))):
        print(f"Sample {i+1}/{min(num_samples, len(folk_samples))}")
        folk_spec = folk_samples[i]
        if isinstance(folk_spec, torch.Tensor):
            folk_spec = folk_spec.to(device)
        else:
            folk_spec = torch.FloatTensor(folk_spec).to(device)
        
        if len(folk_spec.shape) == 2:
            folk_spec = folk_spec.unsqueeze(0)  # [1, 80, 616]
        
        with torch.no_grad():
            initial_output = model(folk_spec)
            initial_prob = torch.sigmoid(initial_output).item()
            initial_jazz = 1 - initial_prob
            initial_folk = initial_prob
        
        transformed, history = folk_to_jazz_transform(
            model=model,
            jazz_spec=folk_spec,
            num_iterations=iterations,
            content_weight=content_weight,
            lr=0.005
        )
        
    
        with torch.no_grad():
            final_output = model(transformed)
            final_prob = torch.sigmoid(final_output).item()
            final_jazz = 1 - final_prob
            final_folk = final_prob
        
        result = {
            'sample_id': i,
            'initial_jazz': initial_jazz,
            'initial_folk': initial_folk,
            'final_jazz': final_jazz,
            'final_folk': final_folk,
            'jazz_improvement': final_jazz - initial_jazz,
            'folk_reduction': initial_folk - final_folk,
            'content_loss_final': history['content_loss'][-1],
            'convergence_iter': None,
            'history': history
        }
        

        jazz_probs = history['jazz_prob']
        folk_probs = history['folk_prob']
        for idx in range(len(folk_probs)):
            if folk_probs[idx] < jazz_probs[idx]:
                result['convergence_iter'] = idx
                result['convergence_folk'] = folk_probs[idx]
                break

        
        results.append(result)
        transformed_total.append(transformed)
        if result['convergence_iter'] is not None:
            print(f"Converged at iteration {result['convergence_iter']}")
    return results,transformed_total

In [28]:
result,transformed=batch_style_transfer_test(model,final)

Sample 1/2207
initial probability - jazz: 0.0001, folk: 0.9999


/var/folders/7j/cmy7fjk976s0k585k8gvs8200000gn/T/ipykernel_36076/4011558351.py:38: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  style_loss = F.mse_loss(logit, torch.tensor([target_logit], device=device))


Iter  20: Jazz=0.7423, Folk=0.2577, StyleLoss=0.8876
jazz: 0.9056, folk: 0.0944
Content Loss: 0.000968
Converged at iteration 4
Sample 2/2207
initial probability - jazz: 0.0024, folk: 0.9976
Iter  20: Jazz=0.8842, Folk=0.1158, StyleLoss=0.0011
jazz: 0.5559, folk: 0.4441
Content Loss: 0.000387
Converged at iteration 1
Sample 3/2207
initial probability - jazz: 0.0002, folk: 0.9998
Iter  20: Jazz=0.9472, Folk=0.0528, StyleLoss=0.7858
jazz: 0.7941, folk: 0.2059
Content Loss: 0.000986
Converged at iteration 4
Sample 4/2207
initial probability - jazz: 0.0000, folk: 1.0000
Iter  20: Jazz=0.9428, Folk=0.0572, StyleLoss=0.6447
jazz: 0.7458, folk: 0.2542
Content Loss: 0.001370
Converged at iteration 5
Sample 5/2207
initial probability - jazz: 0.0000, folk: 1.0000
Iter  20: Jazz=0.9875, Folk=0.0125, StyleLoss=5.6283
jazz: 0.6594, folk: 0.3406
Content Loss: 0.001345
Converged at iteration 5
Sample 6/2207
initial probability - jazz: 0.0000, folk: 1.0000
Iter  20: Jazz=0.9470, Folk=0.0530, StyleLoss

In [47]:
transformed_tensor=torch.stack(transformed).reshape(2207,80,616)

In [56]:
final_transformed=torch.tensor(scaler.inverse_transform(transformed_tensor.reshape(2207,-1))).reshape(2207,80,616)

In [58]:
torch.save(final_transformed,'/Users/robinhu/Desktop/final_to_test/jazz_transformation_total.pt')
